# W01A — Introducción (DDIA Cap. 1) + Entorno local + Evidencia

## Qué haremos hoy
- Entender el “por qué” (DDIA Cap. 1): **Reliability / Scalability / Maintainability**
- Montar un entorno local reproducible (sin Docker).
- Usar SQL solo como **instrumento de evidencia** (no como tema formal aún).

## Mapa del curso (conceptual)
- **Bronze/Raw**: dato tal cual llega (trazabilidad)
- **Silver**: limpieza + tipado + reglas de calidad
- **Gold**: modelado para consulta (hechos/dimensiones/marts/métricas)

## Relación con carpetas (desde ya)
- `data/raw/`  → Bronze/Raw (archivos originales)
- `data/silver/` → outputs intermedios (Parquet/tablas exportadas)
- `data/gold/` → marts/métricas/exports listos para consumo
- `data/exoplanets.duckdb` → “warehouse local” (tablas)
- `artifacts/` → evidencia (JSON de tests, profiling, planes, logs)
- `docs/` → decisiones, runbook, glosario (mantenibilidad)


In [1]:
import os
os.getcwd()

'/home/manuela-rios/Documentos/Física computacional/Manuela-Rios/notebooks'

In [2]:
# 1) Setup (cross-platform)
import sys, platform, hashlib
from pathlib import Path
import duckdb
import os

os.chdir("..")

PROJECT_ROOT = Path(".").resolve()

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR  = DATA_DIR / "raw"
SILVER_DIR = DATA_DIR / "silver"
GOLD_DIR = DATA_DIR / "gold"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
DOCS_DIR = PROJECT_ROOT / "docs"

for d in [RAW_DIR, SILVER_DIR, GOLD_DIR, ARTIFACTS_DIR, DOCS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DB_PATH = DATA_DIR / "exoplanets.duckdb"
con = duckdb.connect(str(DB_PATH))

def show(path: Path):
    return {"exists": path.exists(), "path": str(path), "size_bytes": path.stat().st_size if path.exists() else None}


print("OS:", platform.platform())
print("Python:", sys.version.split()[0])
print("DuckDB:", con.execute("SELECT version()").fetchone()[0])
show(DB_PATH)


OS: Linux-6.17.0-19-generic-x86_64-with-glibc2.39
Python: 3.12.3
DuckDB: v1.4.4


{'exists': True,
 'path': '/home/manuela-rios/Documentos/Física computacional/Manuela-Rios/data/exoplanets.duckdb',
 'size_bytes': 4206592}

## Evidencia mínima: ¿el motor responde?
Esto NO es “clase de SQL”. Es verificar que el motor funciona.

In [3]:
con.execute("SELECT 42 AS answer").fetchall()

[(42,)]

## “Experimento” controlado: una tabla pequeña
La idea: **consulta → evidencia**.

In [4]:
con.execute("CREATE OR REPLACE TABLE demo_numbers(x INTEGER)")
con.execute("INSERT INTO demo_numbers VALUES (1), (2), (3)")
con.execute("SELECT SUM(x) AS s FROM demo_numbers").fetchall()


[(6,)]

## Tu turno (en clase): 2 mini-tablas + 2 métricas
Ejecuta y pega los outputs al final del notebook.

In [5]:
# TU TURNO 1
con.execute("CREATE OR REPLACE TABLE students(name VARCHAR, semester INTEGER)")
con.execute("INSERT INTO students VALUES ('Ana', 7), ('Luis', 8), ('Sofia', 10)")
con.execute("SELECT semester, COUNT(*) n FROM students GROUP BY 1 ORDER BY 1").fetchall()


[(7, 1), (8, 1), (10, 1)]

In [6]:
# TU TURNO 2
con.execute("CREATE OR REPLACE TABLE submissions(name VARCHAR, lab VARCHAR, ok BOOLEAN)")
con.execute("""
INSERT INTO submissions VALUES
  ('Ana', 'W01', TRUE),
  ('Ana', 'W02', FALSE),
  ('Luis','W01', TRUE),
  ('Luis','W02', TRUE),
  ('Sofia','W01', TRUE)
""")
con.execute("SELECT name, COUNT(*) n FROM submissions GROUP BY 1 ORDER BY n DESC").fetchall()


[('Ana', 2), ('Luis', 2), ('Sofia', 1)]

In [7]:
try:
    con.close()
    print("DuckDB connection closed.")
except NameError:
    print("No connection named 'con' in this notebook.")


DuckDB connection closed.


# W01B — Bronze/Raw: ingesta + trazabilidad + sanity checks (DDIA Cap. 1)

## Objetivo
- Garantizar que existe un **Raw** en `data/raw/` con nombre esperado
- Registrar **hash SHA-256** (trazabilidad)
- Consultar el CSV vía **VIEW raw_ps** (sin modificar el archivo)
- Ejecutar sanity checks y guardar evidencia en `artifacts/`

> Importante: En Semana 1 **no enseñamos SQL formal**. SQL aquí es instrumento de evidencia.

In [8]:
# Setup común (cross-platform) + detección de raíz del proyecto
import sys, time, json, hashlib, platform, subprocess
from pathlib import Path
import duckdb

def find_project_root(start: Path) -> Path:
    """Busca hacia arriba una carpeta que contenga `src/` y `data/`.
    Esto evita errores cuando el notebook se ejecuta desde notebooks/."""
    cur = start.resolve()
    for p in [cur] + list(cur.parents):
        if (p / "src").exists() and (p / "data").exists():
            return p
    # fallback: carpeta actual
    return cur

PROJECT_ROOT = find_project_root(Path.cwd())
print("PROJECT_ROOT:", PROJECT_ROOT)

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
DOCS_DIR = PROJECT_ROOT / "docs"
for d in [RAW_DIR, ARTIFACTS_DIR, DOCS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DB_PATH = DATA_DIR / "exoplanets.duckdb"
con = duckdb.connect(str(DB_PATH))

def show(path: Path):
    return {"exists": path.exists(), "path": str(path), "size_bytes": path.stat().st_size if path.exists() else None}

def sha256_file(path: Path, chunk_size: int = 1<<20) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def run_module(mod: str, *args: str):
    """Ejecuta `python -m ...` desde la raíz del proyecto para que `src.*` sea importable."""
    cmd = [sys.executable, "-m", mod, *args]
    print("Running:", " ".join(cmd))
    subprocess.check_call(cmd, cwd=str(PROJECT_ROOT))

print("OS:", platform.platform())
print("Python:", sys.version.split()[0])
print("DuckDB:", con.execute("SELECT version()").fetchone()[0])


PROJECT_ROOT: /home/manuela-rios/Documentos/Física computacional/Manuela-Rios
OS: Linux-6.17.0-19-generic-x86_64-with-glibc2.39
Python: 3.12.3
DuckDB: v1.4.4


## 1) Localizar el archivo Raw

Nombre esperado (por el script oficial del curso): `pscomppars.csv`

**Regla del curso:** el Raw vive en `data/raw/` y NO se edita a mano.


In [9]:
EXPECTED = "pscomppars.csv"
raw_csv = RAW_DIR / EXPECTED

# Si no está con el nombre esperado, buscamos candidatos (para no bloquear la clase)
if not raw_csv.exists():
    candidates = list(RAW_DIR.glob("pscomppars*.csv")) + list(RAW_DIR.glob("*.csv"))
    candidates = [c for c in candidates if c.is_file()]
    if candidates:
        raw_csv = sorted(candidates)[0]
        print(f"No encontré {EXPECTED}. Encontré y usaré: {raw_csv.name}")
        print("   Recomendación: renombra a pscomppars.csv para estandarizar el curso.")
    else:
        raw_csv = None

raw_csv


PosixPath('/home/manuela-rios/Documentos/Física computacional/Manuela-Rios/data/raw/pscomppars.csv')

## 3) Crear VIEW Bronze/Raw

Creamos una vista para consultar el CSV sin modificarlo.


In [10]:
path = str(raw_csv).replace("\\", "/").replace("'", "''")  # Windows-safe
con.execute(f"""
CREATE OR REPLACE VIEW raw_ps AS
SELECT * FROM read_csv_auto('{path}')
""")

In [11]:
con.execute("SELECT COUNT(*) AS n_rows FROM raw_ps").fetchall()


[(6087,)]

## 4) Sanity checks mínimos (evidencia)

Haremos 3 checks:
1) filas y columnas
2) nulos en una columna clave
3) muestra de filas


In [12]:
# Check 1
n_rows = con.execute("SELECT COUNT(*) FROM raw_ps").fetchone()[0]
n_cols = con.execute("SELECT COUNT(*) FROM pragma_table_info('raw_ps')").fetchone()[0]
n_rows, n_cols

(6087, 16)

In [13]:
# 2) nulos en una columna clave típica
con.execute("SELECT COUNT(*) AS null_pl_name FROM raw_ps WHERE pl_name IS NULL").fetchall()

[(0,)]

In [14]:
# Check 3
con.execute("SELECT pl_name, hostname, discoverymethod, disc_year FROM raw_ps WHERE pl_name IS NOT NULL LIMIT 10").fetchall()


[('Kepler-1167 b', 'Kepler-1167', 'Transit', 2016),
 ('Kepler-1740 b', 'Kepler-1740', 'Transit', 2021),
 ('Kepler-1581 b', 'Kepler-1581', 'Transit', 2016),
 ('Kepler-644 b', 'Kepler-644', 'Transit', 2016),
 ('Kepler-1752 b', 'Kepler-1752', 'Transit', 2021),
 ('Kepler-280 c', 'Kepler-280', 'Transit', 2014),
 ('Kepler-1208 b', 'Kepler-1208', 'Transit', 2016),
 ('Kepler-263 c', 'Kepler-263', 'Transit', 2014),
 ('Kepler-1101 b', 'Kepler-1101', 'Transit', 2016),
 ('HD 168746 b', 'HD 168746', 'Radial Velocity', 2002)]

## 5) Guardar evidencia en artifacts (JSON)

Esto es evaluable y reproducible.


In [15]:
artifact = {
  "raw_file": str(raw_csv),
  "raw_sha256": sha256_file(raw_csv),
  "n_rows": n_rows,
  "n_cols": n_cols
}
out = ARTIFACTS_DIR / f"w01b_raw_evidence_{int(time.time())}.json"
out.write_text(json.dumps(artifact, indent=2), encoding="utf-8")
show(out)


{'exists': True,
 'path': '/home/manuela-rios/Documentos/Física computacional/Manuela-Rios/artifacts/w01b_raw_evidence_1775419954.json',
 'size_bytes': 230}

## Reflexión (DDIA Cap.1)
- ¿Qué haría que tu sistema sea “no confiable” aunque el código corra?
- ¿Qué documento te gustaría encontrar si heredas el proyecto de otro equipo?


## Entregable W01
Crea estos archivos (usa `docs/templates/` si existe):
- `docs/decisions_log.md`
- `docs/glossary.md`
- `docs/w01a_run.md`
- `docs/w01b_checks.md`: pega 3 checks (query + resultado).
- `artifacts/w01b_raw_evidence_*.json`: hash + n_rows + n_cols.

## Tarea

- Agregar 1 sanity check extra y documentarlo en docs/w01b_checks.md.

    Ejemplos permitidos:

    - duplicados por pl_name

    - valores negativos en columnas numéricas

    - años fuera de rango (ej. < 1980 o > año actual)

- Registrar una decisión en docs/decisions_log.md (tu formato exacto) sobre trazabilidad:
Ejemplo real:


```markdown
- Fecha: 2025-12-21
- Decisión: Guardar SHA-256 del CSV raw en artifacts por cada ejecución.
- Razón: Detectar cambios invisibles del dato (DDIA reliability/operability).
- Alternativas: Confiar en el nombre del archivo (rechazada), usar solo fecha de descarga (rechazada).
- Evidencia (counts, EXPLAIN, timings): raw_sha256=..., n_rows=..., n_cols=...
```


## SOLUCIÓN TAREA

Para variar el enfoque, en vez de revisar valores negativos, aquí usé un sanity check sobre `disc_year` para detectar años imposibles o fuera del rango esperado. También dejé redactada una decisión de trazabilidad con evidencia concreta.


In [16]:
# Sanity check extra: años de descubrimiento fuera de rango
from datetime import datetime

current_year = datetime.now().year
query_out_of_range = f'''
SELECT
    COUNT(*) AS total_out_of_range,
    MIN(disc_year) AS min_disc_year,
    MAX(disc_year) AS max_disc_year
FROM raw_ps
WHERE disc_year IS NOT NULL
  AND (disc_year < 1980 OR disc_year > {current_year})
'''

out_of_range_summary = con.execute(query_out_of_range).fetchdf()
out_of_range_summary


,total_out_of_range,min_disc_year,max_disc_year
0,0,<NA>,<NA>


In [17]:
# Si existieran registros fuera de rango, los inspeccionamos
query_examples = f'''
SELECT pl_name, hostname, disc_year
FROM raw_ps
WHERE disc_year IS NOT NULL
  AND (disc_year < 1980 OR disc_year > {current_year})
ORDER BY disc_year
LIMIT 10
'''

out_of_range_examples = con.execute(query_examples).fetchdf()
out_of_range_examples


,pl_name,hostname,disc_year


### Texto sugerido para `docs/decisions_log.md`

```markdown
- Fecha: 2026-04-04
- Decisión: Conservar en cada ejecución el hash SHA-256 del archivo raw junto con el número de filas y columnas observadas.
- Razón: Esto mejora la trazabilidad porque permite detectar si el dataset cambió aunque el nombre del archivo siga siendo el mismo.
- Alternativas: confiar solo en el nombre del CSV (rechazada), registrar solo la fecha de descarga (rechazada).
- Evidencia (counts, EXPLAIN, timings): raw_sha256 = <pegar valor>, n_rows = <pegar valor>, n_cols = <pegar valor>.
```
